In [1]:
import json
import torch.nn as nn
import torch
from model.Transformer import Transformer,encoder_stack,decoder_stack
from model.Transformer_block import encoder_block,decoder_block
from Data.dataset import Dataset

with open("src_vocab.json", "r") as f:
    src_vocab = json.load(f)

with open("tgt_vocab.json", "r") as f:
    tgt_vocab = json.load(f)

embed_dim = 64

encoder_blocks= [encoder_block(embed_dim,1,128,eps=1e-5) for _ in range(2)]
decoder_blocks= [decoder_block(embed_dim,1,128,eps=1e-5) for _ in range(2)]

enc_stack= encoder_stack(encoder_blocks)
dec_stack= decoder_stack(decoder_blocks)

model= Transformer(enc_stack, dec_stack, embed_dim, len(src_vocab), len(tgt_vocab))

device= torch.device("cuda" if torch.cuda.is_available() else "cpu")

model= model.to(device)

In [2]:
from datasets import load_dataset
import re
from torch.utils.data import DataLoader
from Data.dataloader import collate

def clean(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Zà-ÿ\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

ds = load_dataset("Helsinki-NLP/opus-100", "en-it")

data_pair = []

for sample in ds["train"]:
    en= clean(sample["translation"]["en"])
    it= clean(sample["translation"]["it"])

    data_pair.append((en, it))


dataset= Dataset(data_pair,src_vocab_path="src_vocab.json",tgt_vocab_path="tgt_vocab.json", src_vocab=src_vocab,tgt_vocab=tgt_vocab)


train_loader= DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=collate)



{'src': tensor([[ 72345, 182831,      0,      0],
        [ 78050,  97223,  98613,  93247],
        [ 75494,   8937, 184931,      0]]), 'tgt_in': tensor([[     2,  39480, 139199,      0,      0],
        [     2,  10793, 100319, 128921, 120887],
        [     2,  43055, 207261,      0,      0]]), 'tgt_out': tensor([[ 39480, 139199,      3,      0,      0],
        [ 10793, 100319, 128921, 120887,      3],
        [ 43055, 207261,      3,      0,      0]])}


In [ ]:
scaler= torch.amp.GradScaler("cuda")

epochs=10
criterion= torch.nn.CrossEntropyLoss()
optimizer= torch.optim.AdamW(model.parameters(), lr=1e-5)

for epoch in range(epochs):

    running_loss=0
    model.train()

    for batch_idx,batch in enumerate(train_loader):

        optimizer.zero_grad()

        src = batch["src"].to(device)
        tgt_in = batch["tgt_in"].to(device)
        tgt_out = batch["tgt_out"].to(device)

        with torch.autocast(device_type="cuda", dtype=torch.float16):

            output= model(src,tgt_in)
            loss= criterion(output.reshape(-1,len(tgt_vocab)),tgt_out.reshape(-1))


        scaler.scale(loss).backward()

        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)

    print(f"Epoch [{epoch+1}/{epochs}] | Loss: {epoch_loss:.4f}")






In [ ]:
print(model)

Transformer(
  (src_embedding): Embedding(187024, 64)
  (tgt_embedding): Embedding(239423, 64)
  (src_pos): PositionalEncoding()
  (tgt_pos): PositionalEncoding()
  (fc_out): Linear(in_features=64, out_features=239423, bias=True)
  (encoder): encoder_stack(
    (layers): ModuleList(
      (0-1): 2 x encoder_block(
        (ffn1): FFN(
          (W1): Linear(in_features=64, out_features=128, bias=True)
          (W2): Linear(in_features=128, out_features=64, bias=True)
        )
        (ln1): LayerNorm()
        (ln2): LayerNorm()
        (MHA): MultiHeadAttention(
          (q_proj): Linear(in_features=64, out_features=64, bias=True)
          (k_proj): Linear(in_features=64, out_features=64, bias=True)
          (v_proj): Linear(in_features=64, out_features=64, bias=True)
          (Wo_proj): Linear(in_features=64, out_features=64, bias=True)
        )
      )
    )
  )
  (decoder): decoder_stack(
    (layers): ModuleList(
      (0-1): 2 x decoder_block(
        (ffn1): FFN(
        